*Notebook Last Validated: 2026-09-25*

# Rhino SDK - Resource Utilization Notebook

Checks whether the workgroups collaborating on a Rhino FCP project currently have enough
free CPU, memory, storage, and GPU capacity to safely run a job.

For each workgroup, resource usage is sampled several times over a short window (rather than
a single instant) and reduced to worst-case ("peak") values, then compared against
configurable thresholds. Sampling and parsing helpers live in [utils.py](utils.py).

**Steps:**
1. Log in to Rhino and load the target project's collaborating workgroups.
2. Sample each workgroup's live resource usage and print a summary.
3. Check the sampled usage against thresholds to decide if it's safe to proceed.

All four thresholds in step 4 (CPU, RAM, GPU compute, GPU memory) are independently optional - leave any of them unset (`None`) to skip that check entirely, e.g. if your job doesn't need a GPU at all.

In [2]:
import rhino_health as rh
from getpass import getpass
from utils import sample_agent_resources, parse_system_resources, print_system_resources, check_resource_thresholds
print("Imports completed.")

Imports completed.


## 1. Log in

Enter your Rhino username when prompted and your password via the hidden `getpass` prompt.

In [3]:
USERNAME = "<YOUR_USERNAME>" # Update this to your Rhino FCP username
print("Logging in...")
session = rh.login(USERNAME, getpass())
print("Successfully logged in!")

Logging in...


 ········


Successfully logged in!


## 2. Load the project and its collaborating workgroups

Set `PROJECT` to the name of the project to check, then list every workgroup collaborating on it.

**Edge cases handled below:**
- **No project with that name** - raises a clear error rather than failing later with a confusing `AttributeError` on `None`.
- **Multiple projects with that name** - project names aren't guaranteed unique, so this is possible. `session.project.get_project_by_name` silently picks the most recently created match with no warning, which could point you at the wrong project without you noticing. The cell below checks for this and prints a warning so you can double check (or rename the project to something unique) if that's not what you intended.

In [4]:
from rhino_health.lib.endpoints.endpoint import NameFilterMode

PROJECT = "DO-435 Test" # Update this to the name of the project you want to check resource utilization for. The project must be one that your account has access to, and the name must match exactly (case-sensitive).

matches = session.project.search_for_projects_by_name(PROJECT, NameFilterMode.EXACT)
if not matches:
    raise ValueError(
        f"No project found named '{PROJECT}'. Double-check the spelling and capitalization "
        f"(matching is case-sensitive and exact), and that your account has access to it."
    )
if len(matches) > 1:
    newest = max(matches, key=lambda p: p.created_at)
    print(
        f"WARNING: {len(matches)} projects are named '{PROJECT}'. \nUsing the most recently "
        f"created one (uid={newest.uid}, created {newest.created_at}). \nIf that's not the "
        f"project you meant, rename it to something unique or pick the right uid from "
        f"'matches' above and load it explicitly."
    )

project = session.project.get_project_by_name(PROJECT)
print(f"\nProject: <{project.name}>")
workgroups = project.collaborating_workgroups

if not workgroups:
    raise ValueError(
        f"Project '{project.name}' has no collaborating workgroups. There's nothing for the "
        f"rest of this notebook to check - add a collaborating workgroup to the project first."
    )

print(f"Associated Workgroups:")
for workgroup in workgroups:
    print(f"    <{workgroup.name}>")


Project: <DO-435 Test>
Associated Workgroups:
    <Rhino Health>


## 3. Sample and summarize resource usage

For each workgroup, take 10 samples of live resource usage over 30 seconds, reduce them to
peak (worst-case) values, and print a summary. 

Sampling repeatedly (rather than checking once) matters because usage is a moving target - a single snapshot could catch a workgroup at an unusually quiet or busy moment; taking the *peak* across several samples gives a more honest worst-case picture of whether it'll have room for your job.

**`sample_agent_resources(project, workgroup_uid, num_samples, duration_sec)`**
- `project` — the Rhino SDK `Project` object loaded in step 2.
- `workgroup_uid` — UID of the workgroup being sampled (`workgroup.uid`).
- `num_samples` (`10`) — how many times to query the workgroup's agent resources.
- `duration_sec` (`30`) — total time in seconds over which the `num_samples` queries are spread; sleeps evenly between calls.
- **Why adjust these:** raise both for a longer-running or more resource-intensive job, to get a
  better sense of sustained peak load rather than a brief blip (e.g. `num_samples=20`,
  `duration_sec=120` to sample over 2 minutes). Lower them for a quick sanity check when you're
  not too worried about transient spikes - just be aware fewer/closer-together samples are more
  likely to miss a real peak.
- Returns a list of raw `SystemResources` objects, one per sample.

**`parse_system_resources(resources)`**
- `resources` — either a single `SystemResources` object or, as used here, the list returned by `sample_agent_resources`. When given a list, each sample is parsed individually and then reduced to peak values per metric (max for usage %/GB-used metrics, min for free-GB metrics).
- Returns the parsed metrics dict (`cpu`, `memory`, `storage`, `gpu`).

**`print_system_resources(parsed, wg_name)`**
- `parsed` — the dict returned by `parse_system_resources`.
- `wg_name` — display name for the summary header (`workgroup.name`).

In [5]:
for workgroup in workgroups:
    wg_uid = workgroup.uid
    print(f"Sampling Workgroup <{workgroup.name}> : (10 samples over ~30s)...")
    samples = sample_agent_resources(project, wg_uid, num_samples=10, duration_sec=30)
    parsed = parse_system_resources(samples)
    print_system_resources(parsed, workgroup.name)
    print()

Sampling Workgroup <Rhino Health> : (10 samples over ~30s)...
=== Resource Summary for Workgroup: Rhino Health ===
  CPU Used:          10.9%
  Memory Used:       4.3 GB / 15.33 GB (28.07%)
  Storage Used:      86.08 GB / 241.13 GB (35.7%)



## 4. Check against thresholds

Re-samples each workgroup (fresh data, right before you'd actually kick off your job - not
reusing step 3's samples, since usage can change in the time it takes to review step 3's
output) and checks the peak values against the thresholds below. 

A workgroup fails if its peak CPU usage, free RAM, GPU compute usage, or GPU memory usage falls outside the given limits.

Adjust the threshold values below to match what your job actually needs - see "Why adjust" under each parameter.

**`check_resource_thresholds(parsed, cpu_threshold, ram_free_gb_threshold, gpu_threshold, gpu_memory_threshold)`**
- `parsed` — the peak metrics dict from `parse_system_resources` for this workgroup.
- `cpu_threshold` (`40.0` in the cell below; defaults to `None`, meaning "don't check CPU") — max allowable CPU usage, in percent; fails if peak `cpu.used_pct` exceeds this.
  - **Why adjust:** lower it for CPU-bound jobs where you need headroom (e.g. `20.0` if your job itself will be CPU-intensive and the workgroup can't be busy already); raise it (or leave `None`) if your job is lightweight or GPU-bound and CPU contention doesn't matter much.
- `ram_free_gb_threshold` (`10.0` in the cell below; defaults to `None`) — minimum required free RAM, in GB; fails if peak-free `memory.free_gb` drops below this.
  - **Why adjust:** set this to at least your job's expected peak memory footprint, plus some margin - e.g. `32.0` for a job that loads a large dataset or model into memory.
- `gpu_threshold` (`50.0` in the cell below; defaults to `None`) — max allowable GPU **compute** usage, in percent; passes as long as at least one GPU device's `compute_used_pct` is at or under this.
  - **Why adjust:** lower it (e.g. `10.0`) if your job needs a mostly-idle GPU to itself; leave it higher, or `None`, if your job doesn't need a GPU at all or can tolerate sharing one.
- `gpu_memory_threshold` (`5.0` in the cell below; defaults to `None`) — max allowable GPU **memory** usage, in percent (this is a usage percentage, not a GB value - the platform doesn't expose absolute VRAM capacity per device, only percent-used); passes as long as at least one GPU device's `memory_used_pct` is at or under this.
  - **Why adjust:** lower it for jobs that need most of a GPU's VRAM free (e.g. large model training/fine-tuning); a value like `5.0` means "at least one GPU must be under 5% memory-used" - a fairly strict bar.

- Any of the four thresholds left as `None` skips that check entirely for that metric.

- **Edge case:** if `gpu_threshold` or `gpu_memory_threshold` is set to a *nonzero* value and a workgroup has **no GPU at all**, that workgroup automatically fails the whole check - there's no GPU device to evaluate against. Leave both as `None` if your job doesn't require a GPU, so GPU-less workgroups aren't unfairly disqualified.

- Returns a dict, not just a bool: `{"passed": bool, "failures": [...]}`. `passed` is `True` only if every threshold that was provided is satisfied. `failures` lists every threshold that was NOT met (not just the first one hit) - each entry names the metric, the actual peak value, the threshold, and a `detail` string spelling out exactly by how much it missed (e.g. "CPU usage peaked at 62.3%, 22.3 percentage points over the 40.0% limit"). The cell below prints this detail for any workgroup that fails.

In [6]:
per_workgroup_check = []
for workgroup in workgroups:
    wg_uid = workgroup.uid
    print(f"Re-sampling Workgroup <{workgroup.name}> : (10 samples over ~30s)...")
    samples = sample_agent_resources(project, wg_uid, num_samples=10, duration_sec=30)
    parsed = parse_system_resources(samples)
    result = check_resource_thresholds(
        parsed=parsed,
        cpu_threshold=40.0,
        ram_free_gb_threshold=10.0,
        gpu_threshold=50.0,
        gpu_memory_threshold=5.0
    )
    per_workgroup_check.append(result["passed"])
    status = "PASS ✅" if result["passed"] else "FAIL ❌"
    print(f"  <{workgroup.name}>: {status}")
    for failure in result["failures"]:
        print(f"      ✗ {failure['metric']}: {failure['detail']}")

if all(per_workgroup_check):
    print("\nAll workgroups have enough resources. You can run your code based on the thresholds you provided.")
else:
    print("\nOne or more workgroups do not meet the specified resource thresholds. Please review before proceeding.")

Re-sampling Workgroup <Rhino Health> : (10 samples over ~30s)...
  <Rhino Health>: FAIL ❌
      ✗ GPU availability: gpu_threshold and/or gpu_memory_threshold was set, but this workgroup has no GPU at all.

One or more workgroups do not meet the specified resource thresholds. Please review before proceeding.
